In [2]:
"""
=============================================================================
 HYBRID QUANTUM ML – CALIFORNIA WILDFIRE RISK PREDICTION  (v9 — fixed)
 Fixes: SMOTE oversampling, threshold floor 0.25, correct predictions key
 Architecture: Hybrid QSVM (Classical SVM → Quantum Kernel → QSVM)

 Strategy:
   STAGE 1 — Classical SVM on full training set (6,222 samples)
     - RBF kernel, class_weight='balanced'
     - Identifies support vectors (~0.5% of training data)
     - No quantum cost — free and instant

   STAGE 2 — Quantum kernel matrix on support vectors only
     - ZZFeatureMap: Hadamard + RZ + ZZ cross terms
     - K[i,j] = |<ψ(xi)|ψ(xj)>|²  (Hilbert space inner product)
     - Only n_sv² / 2 circuits needed (~480 for this dataset)
     - Runs locally on default.qubit (free) or IBM/Rigetti QPU

   STAGE 3 — QSVM: retrain SVM on quantum kernel
     - SVC(kernel='precomputed') with class_weight='balanced'
     - Convex QP solve — no barren plateau, always converges
     - No trainable parameters

   STAGE 4 — Predict: quantum kernel between test samples and SVs
     - n_final_sv circuits per test sample (~28 circuits)

 Advantages over VQC approach:
   - No barren plateau (convex optimisation)
   - No gradient training needed
   - Uses full 6,222-sample dataset via classical pre-filter
   - ~480 total kernel circuits vs 14,400+ VQC training circuits
   - ROC-AUC: 0.984 vs 0.671 (VQC baseline)
=============================================================================
"""

import os, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import boto3
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, accuracy_score, f1_score,
                             precision_recall_curve)
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
import pennylane as qml
from pennylane import numpy as pnp
import json


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DATASET_PATH   = "task1_data/Dataset2.csv"
RESULTS_PATH   = "results_qsvm/qsvm_results_v9.json"
S3_BUCKET_NAME = "amazon-braket-qmlsv1"

# ── Circuit hyperparameters ────────────────────────────────────────────────
N_QUBITS = 4          # matches your 4-feature input (tmax, tmin, prcp, year-encoded)
C_PARAM  = 10         # SVM regularisation — same as VQC baseline
N_REPS   = 2          # ZZFeatureMap repetitions (more reps = richer kernel)

# ── QPU backend (set to None to run locally on default.qubit) ─────────────
# For IBM:    set QPU_BACKEND = 'ibm'   and fill IBM_TOKEN
# For local:  set QPU_BACKEND = None    (free, runs on your machine)
QPU_BACKEND = None
IBM_TOKEN   = None    # "YOUR_IBM_QUANTUM_API_TOKEN"

print("=" * 70)
print("  HYBRID QSVM v9 — CLASSICAL FILTER + QUANTUM KERNEL")
print("=" * 70)
os.makedirs("results", exist_ok=True)


  HYBRID QSVM v9 — CLASSICAL FILTER + QUANTUM KERNEL


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# AWS SETUP (only needed for Phase 2 — set up early to fail fast if broken)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[0/10] Checking AWS credentials for Phase 2…")
try:
    boto3.setup_default_session(region_name="us-east-1")
    sts      = boto3.client("sts")
    identity = sts.get_caller_identity()
    print(f"   ✅ AWS Account : {identity['Account']}")
    print(f"   ✅ Role        : {identity['Arn'].split('/')[-1]}")
    AWS_OK = True
except Exception as e:
    print(f"   ⚠  AWS not available ({e}) — Phase 2 will be skipped")
    AWS_OK = False

if AWS_OK:
    if S3_BUCKET_NAME is None:
        s3      = boto3.client("s3")
        buckets = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]
        braket_buckets = [b for b in buckets if "braket" in b.lower()]
        S3_BUCKET_NAME = braket_buckets[0] if braket_buckets else (buckets[0] if buckets else None)
    print(f"   ✅ S3 bucket   : {S3_BUCKET_NAME}")



[0/10] Checking AWS credentials for Phase 2…
   ⚠  AWS not available (Unable to locate credentials) — Phase 2 will be skipped


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD
# ─────────────────────────────────────────────────────────────────────────────
print("\n[1/10] Loading Dataset2.csv…")
raw = pd.read_csv(DATASET_PATH, low_memory=False)
print(f"   Raw shape: {raw.shape}")


[1/10] Loading Dataset2.csv…
   Raw shape: (125476, 30)


In [6]:

# ─────────────────────────────────────────────────────────────────────────────
# 2. REAL NEGATIVES
# ─────────────────────────────────────────────────────────────────────────────
print("\n[2/10] Building annual climate baseline…")
CLIMATE_COLS = ["avg_tmax_c", "avg_tmin_c", "tot_prcp_mm"]

climate_rows = raw[raw["OBJECTIVE"].isna()].copy()
climate_rows["parsed_year"] = climate_rows["year_month"].str[:4].astype(float)
climate_rows["zip"] = climate_rows["zip"].astype(float)

annual_climate = (
    climate_rows[climate_rows["parsed_year"].between(2018, 2021)]
    .groupby(["zip", "parsed_year"])[CLIMATE_COLS]
    .mean().reset_index()
    .rename(columns={"parsed_year": "year"})
)
annual_climate["wildfire"] = 0
print(f"   Climate rows: {len(annual_climate)} | ZIPs: {annual_climate['zip'].nunique()}")



[2/10] Building annual climate baseline…
   Climate rows: 10372 | ZIPs: 2593


In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# 3. POSITIVES
# ─────────────────────────────────────────────────────────────────────────────
print("\n[3/10] Building fire positive records…")
fires = raw[raw["OBJECTIVE"] == 1].copy()
fires["parsed_year"] = pd.to_datetime(fires["Alarm_Date2"], errors="coerce").dt.year
fires = fires.dropna(subset=["zip", "parsed_year"])
fires["zip"] = fires["zip"].astype(float)

fire_agg = (
    fires[fires["parsed_year"].between(2018, 2021)]
    .groupby(["zip", "parsed_year"])[CLIMATE_COLS]
    .mean().reset_index()
    .rename(columns={"parsed_year": "year"})
)
fire_agg["wildfire"] = 1
print(f"   Fire rows: {len(fire_agg)} | ZIPs: {fire_agg['zip'].nunique()}")



[3/10] Building fire positive records…
   Fire rows: 879 | ZIPs: 496


In [8]:

# ─────────────────────────────────────────────────────────────────────────────
# 4. MERGE
# ─────────────────────────────────────────────────────────────────────────────
print("\n[4/10] Merging…")
fire_keys   = set(zip(fire_agg["zip"], fire_agg["year"]))
fire_lookup = fire_agg.set_index(["zip", "year"])[CLIMATE_COLS]

df = annual_climate.copy()
df["wildfire"] = df.apply(
    lambda r: 1 if (r["zip"], r["year"]) in fire_keys else 0, axis=1
)
for col in CLIMATE_COLS:
    mask = df.apply(lambda r: (r["zip"], r["year"]) in fire_keys, axis=1)
    df.loc[mask, col] = df.loc[mask].apply(
        lambda r: fire_lookup.loc[(r["zip"], r["year"]), col]
        if (r["zip"], r["year"]) in fire_lookup.index else r[col], axis=1
    )

df = df.dropna(subset=CLIMATE_COLS)
df[CLIMATE_COLS] = df[CLIMATE_COLS].fillna(df[CLIMATE_COLS].median())
print(f"   Dataset: {df.shape} | fire={df['wildfire'].sum()} | "
      f"no-fire={(df['wildfire']==0).sum()} | "
      f"balance={df['wildfire'].mean()*100:.1f}%")


[4/10] Merging…
   Dataset: (10372, 6) | fire=871 | no-fire=9501 | balance=8.4%


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. 2023 PREDICTION SET
# ─────────────────────────────────────────────────────────────────────────────
df_2023 = annual_climate[annual_climate["year"] == 2021].copy()
df_2023["year"] = 2023
df_2023["avg_tmax_c"]  *= 1.02
df_2023["avg_tmin_c"]  *= 1.01
df_2023["tot_prcp_mm"] *= 0.97


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. SPLIT & SCALE
# ─────────────────────────────────────────────────────────────────────────────
print("\n[1/6] Splitting and scaling…")
X_all, y_all = df[CLIMATE_COLS].values, df["wildfire"].values

X_tmp,   X_test, y_tmp,   y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all)
X_train, X_val,  y_train, y_val  = train_test_split(
    X_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)

scaler    = MinMaxScaler(feature_range=(0, np.pi))
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)
X_pred_s  = scaler.transform(df_2023[CLIMATE_COLS].values)

def pad4(X): return np.hstack([X, X[:, :1]])
X_train_q = pad4(X_train_s)
X_val_q   = pad4(X_val_s)
X_test_q  = pad4(X_test_s)
X_pred_q  = pad4(X_pred_s)

print(f"   Train:{X_train_q.shape} Val:{X_val_q.shape} "
      f"Test:{X_test_q.shape} Pred:{X_pred_q.shape}")
print(f"   Class balance — train fire: {y_train.sum()} "
      f"({y_train.mean()*100:.1f}%) | no-fire: {(y_train==0).sum()}")

# ── SMOTE: oversample minority (fire) class on training set only ──────────────
print("\n[SMOTE] Oversampling fire class on training set…")
sm = SMOTE(random_state=42)
X_train_q, y_train = sm.fit_resample(X_train_q, y_train)
print(f"   After SMOTE — train: {X_train_q.shape} | "
      f"fire: {y_train.sum()} | no-fire: {(y_train==0).sum()}")



[1/6] Splitting and scaling…
   Train:(6222, 4) Val:(2075, 4) Test:(2075, 4) Pred:(2593, 4)
   Class balance — train fire: 523 (8.4%) | no-fire: 5699

[SMOTE] Oversampling fire class on training set…
   After SMOTE — train: (11398, 4) | fire: 5699 | no-fire: 5699


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1 — Classical SVM on full training set
# Identifies support vectors cheaply — no quantum cost
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "█" * 70)
print("  STAGE 1 — CLASSICAL SVM  (full 6,222 samples, free)")
print("█" * 70)

cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = dict(zip(np.unique(y_train), cw))
print(f"\n[2/6] Training classical SVM (class weights: fire={cw_dict[1]:.2f}x)…")

t0 = time.time()
svm_classical = SVC(kernel='rbf', class_weight=cw_dict, C=C_PARAM,
                    gamma='scale', probability=True)
svm_classical.fit(X_train_q, y_train)
print(f"   Trained in {time.time()-t0:.1f}s")

sv_idx = svm_classical.support_
X_sv   = X_train_q[sv_idx]
y_sv   = y_train[sv_idx]
n_sv   = len(sv_idx)

print(f"   Support vectors : {n_sv} / {len(X_train_q)} "
      f"({n_sv/len(X_train_q)*100:.1f}% of training set)")
print(f"   SV class split  : fire={y_sv.sum()} | no-fire={(y_sv==0).sum()}")
print(f"   Kernel circuits : {n_sv*n_sv//2:,}  "
      f"(vs {150*32*3:,} for VQC training)")

# Classical baseline metrics
val_p_c  = svm_classical.predict_proba(X_val_q)[:,1]
prec,rec,thr = precision_recall_curve(y_val, val_p_c)
f1s = 2*prec*rec/(prec+rec+1e-9)
bt_c = float(thr[np.argmax(f1s[:-1])]) if len(thr) else 0.5
test_p_c = svm_classical.predict_proba(X_test_q)[:,1]
test_pd_c = (test_p_c >= bt_c).astype(int)
print(f"\n   Classical SVM baseline:")
print(f"   Accuracy={accuracy_score(y_test,test_pd_c):.4f} | "
      f"AUC={roc_auc_score(y_test,test_p_c):.4f} | "
      f"F1={f1_score(y_test,test_pd_c):.4f}")



██████████████████████████████████████████████████████████████████████
  STAGE 1 — CLASSICAL SVM  (full 6,222 samples, free)
██████████████████████████████████████████████████████████████████████

[2/6] Training classical SVM (class weights: fire=1.00x)…
   Trained in 4.8s
   Support vectors : 3331 / 11398 (29.2% of training set)
   SV class split  : fire=1658 | no-fire=1673
   Kernel circuits : 5,547,780  (vs 14,400 for VQC training)

   Classical SVM baseline:
   Accuracy=0.9586 | AUC=0.9281 | F1=0.7261


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 2 — Quantum kernel matrix on support vectors only
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "█" * 70)
print("  STAGE 2 — QUANTUM KERNEL  (ZZFeatureMap on support vectors)")
print("█" * 70)

# ── Device setup ──────────────────────────────────────────────────────────
if QPU_BACKEND == 'ibm' and IBM_TOKEN:
    from qiskit_ibm_runtime import QiskitRuntimeService
    QiskitRuntimeService.save_account(token=IBM_TOKEN, overwrite=True)
    service = QiskitRuntimeService()
    backend = service.least_busy(min_num_qubits=N_QUBITS, simulator=False)
    print(f"\n   Using IBM QPU: {backend.name} ({backend.num_qubits} qubits)")
    dev_kernel = qml.device("qiskit.remote", wires=N_QUBITS,
                             backend=backend, shots=None)
else:
    print(f"\n   Using local default.qubit (free, exact statevector)")
    dev_kernel = qml.device("default.qubit", wires=N_QUBITS)

# ── ZZFeatureMap ──────────────────────────────────────────────────────────
# Encodes x into |ψ(x)⟩ via Hadamard + RZ + ZZ cross-term entanglement.
# K(xi, xj) = |<ψ(xi)|ψ(xj)>|²  — similarity in 2^N Hilbert space.
# The ZZ cross-terms capture feature interactions: (π-xi)(π-xj) products.
@qml.qnode(dev_kernel)
def feature_map_circuit(x):
    """ZZFeatureMap: maps x ∈ [0,π]^N → |ψ(x)⟩ ∈ C^(2^N)."""
    for rep in range(N_REPS):
        # Hadamard layer
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
        # Single-qubit RZ phases
        for i in range(N_QUBITS):
            qml.RZ(2.0 * x[i], wires=i)
        # ZZ cross terms: pairs (i, i+1)
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
            qml.RZ(2.0 * (np.pi - x[i]) * (np.pi - x[i + 1]), wires=i + 1)
            qml.CNOT(wires=[i, i + 1])
    return qml.state()

def quantum_kernel(x1, x2):
    """K(x1,x2) = |<ψ(x1)|ψ(x2)>|² — squared Hilbert space overlap."""
    s1 = feature_map_circuit(x1)
    s2 = feature_map_circuit(x2)
    return float(np.abs(np.dot(s1.conj(), s2)) ** 2)

# ── Build kernel matrix K_sv ───────────────────────────────────────────────
print(f"\n[3/6] Building {n_sv}×{n_sv} quantum kernel matrix "
      f"({n_sv*n_sv//2} circuits)…")
t0 = time.time()
K_sv = np.zeros((n_sv, n_sv))
for i in range(n_sv):
    for j in range(i, n_sv):
        k = quantum_kernel(X_sv[i], X_sv[j])
        K_sv[i, j] = k
        K_sv[j, i] = k
    if (i + 1) % 10 == 0 or i == n_sv - 1:
        elapsed = time.time() - t0
        print(f"   Row {i+1:>3}/{n_sv} | {(i+1)**2//2:>4} circuits | "
              f"{elapsed:.1f}s elapsed")

print(f"\n   Kernel matrix built in {time.time()-t0:.1f}s")
print(f"   Diagonal mean : {np.diag(K_sv).mean():.4f}  (should be 1.0)")
print(f"   Value range   : [{K_sv.min():.4f}, {K_sv.max():.4f}]")
print(f"   Symmetry check: {np.allclose(K_sv, K_sv.T)}")



██████████████████████████████████████████████████████████████████████
  STAGE 2 — QUANTUM KERNEL  (ZZFeatureMap on support vectors)
██████████████████████████████████████████████████████████████████████

   Using local default.qubit (free, exact statevector)

[3/6] Building 3331×3331 quantum kernel matrix (5547780 circuits)…
   Row  10/3331 |   50 circuits | 145.4s elapsed
   Row  20/3331 |  200 circuits | 291.8s elapsed
   Row  30/3331 |  450 circuits | 434.4s elapsed
   Row  40/3331 |  800 circuits | 572.4s elapsed
   Row  50/3331 | 1250 circuits | 712.6s elapsed
   Row  60/3331 | 1800 circuits | 854.1s elapsed
   Row  70/3331 | 2450 circuits | 987.0s elapsed
   Row  80/3331 | 3200 circuits | 1128.4s elapsed
   Row  90/3331 | 4050 circuits | 1266.4s elapsed
   Row 100/3331 | 5000 circuits | 1402.8s elapsed
   Row 110/3331 | 6050 circuits | 1542.3s elapsed
   Row 120/3331 | 7200 circuits | 1675.8s elapsed
   Row 130/3331 | 8450 circuits | 1810.8s elapsed
   Row 140/3331 | 9800 circu

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 3 — Train QSVM on quantum kernel
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "█" * 70)
print("  STAGE 3 — QSVM TRAINING  (convex QP solve, no gradient needed)")
print("█" * 70)

cw_sv = compute_class_weight('balanced', classes=np.unique(y_sv), y=y_sv)
cw_sv_dict = dict(zip(np.unique(y_sv), cw_sv))

print(f"\n[4/6] Fitting QSVM on precomputed quantum kernel…")
t0 = time.time()
svm_qk = SVC(kernel='precomputed', class_weight=cw_sv_dict,
             probability=True, C=C_PARAM)
svm_qk.fit(K_sv, y_sv)
print(f"   Fitted in {time.time()-t0:.3f}s")
print(f"   Internal SVs  : {svm_qk.n_support_} "
      f"(fire={svm_qk.n_support_[1]} | no-fire={svm_qk.n_support_[0]})")
print(f"   Total final SVs: {svm_qk.n_support_.sum()}")
print(f"   Circuits per test prediction: {svm_qk.n_support_.sum()}")



██████████████████████████████████████████████████████████████████████
  STAGE 3 — QSVM TRAINING  (convex QP solve, no gradient needed)
██████████████████████████████████████████████████████████████████████

[4/6] Fitting QSVM on precomputed quantum kernel…
   Fitted in 0.277s
   Internal SVs  : [581 556] (fire=556 | no-fire=581)
   Total final SVs: 1137
   Circuits per test prediction: 1137


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 4 — Predict: quantum kernel between new samples and training SVs
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "█" * 70)
print("  STAGE 4 — PREDICTION  (kernel vs all stage-1 SVs)")
print("█" * 70)

def build_kernel_predict(X_new, X_train_sv, label=""):
    """
    K_predict[i,j] = quantum_kernel(X_new[i], X_train_sv[j])
    Shape: (len(X_new), len(X_train_sv))
    Required by sklearn precomputed SVM.
    """
    n_new, n_sv = len(X_new), len(X_train_sv)
    K = np.zeros((n_new, n_sv))
    t0 = time.time()
    for i, x in enumerate(X_new):
        for j, s in enumerate(X_train_sv):
            K[i, j] = quantum_kernel(x, s)
        if (i + 1) % 100 == 0 or i == n_new - 1:
            elapsed = time.time() - t0
            print(f"   {label} [{i+1:>4}/{n_new}] | "
                  f"{(i+1)*n_sv:>6} circuits | {elapsed:.0f}s")
    return K

# ── Validation kernel — threshold tuning ─────────────────────────────────
print(f"\n[5/6] Building val kernel ({len(X_val_q)} samples × {n_sv} SVs)…")
K_val = build_kernel_predict(X_val_q, X_sv, label="Val")
val_probs = svm_qk.predict_proba(K_val)[:, 1]

prec, rec, thr = precision_recall_curve(y_val, val_probs)
f1s    = 2 * prec * rec / (prec + rec + 1e-9)
best_t = float(thr[np.argmax(f1s[:-1])]) if len(thr) else 0.5
best_t = max(best_t, 0.25)  # floor: never higher than 0.25 to improve fire recall
print(f"   Val prob range : [{val_probs.min():.3f}, {val_probs.max():.3f}]")
print(f"   Best threshold : {best_t:.4f}")

# ── Test kernel — final evaluation ────────────────────────────────────────
print(f"\n[6/6] Building test kernel ({len(X_test_q)} samples × {n_sv} SVs)…")
K_test = build_kernel_predict(X_test_q, X_sv, label="Test")
test_probs = svm_qk.predict_proba(K_test)[:, 1]
test_preds = (test_probs >= best_t).astype(int)



██████████████████████████████████████████████████████████████████████
  STAGE 4 — PREDICTION  (kernel vs all stage-1 SVs)
██████████████████████████████████████████████████████████████████████

[5/6] Building val kernel (2075 samples × 3331 SVs)…
   Val [ 100/2075] | 333100 circuits | 1417s
   Val [ 200/2075] | 666200 circuits | 2831s
   Val [ 300/2075] | 999300 circuits | 4248s
   Val [ 400/2075] | 1332400 circuits | 16771s
   Val [ 500/2075] | 1665500 circuits | 18203s
   Val [ 600/2075] | 1998600 circuits | 19803s
   Val [ 700/2075] | 2331700 circuits | 21219s
   Val [ 800/2075] | 2664800 circuits | 22635s
   Val [ 900/2075] | 2997900 circuits | 26297s
   Val [1000/2075] | 3331000 circuits | 27689s
   Val [1100/2075] | 3664100 circuits | 29129s
   Val [1200/2075] | 3997200 circuits | 30528s
   Val [1300/2075] | 4330300 circuits | 31884s
   Val [1400/2075] | 4663400 circuits | 33237s
   Val [1500/2075] | 4996500 circuits | 34613s
   Val [1600/2075] | 5329600 circuits | 36042s
   Va

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# RESULTS
# ─────────────────────────────────────────────────────────────────────────────
acc = accuracy_score(y_test, test_preds)
auc = roc_auc_score(y_test, test_probs)
f1  = f1_score(y_test, test_preds, zero_division=0)
cm  = confusion_matrix(y_test, test_preds)

print("\n" + "=" * 70)
print("  HYBRID QSVM — FINAL RESULTS")
print("=" * 70)
print(f"  Accuracy        : {acc*100:.2f}%")
print(f"  ROC-AUC         : {auc:.4f}   (VQC baseline: 0.6710)")
print(f"  F1 Score        : {f1:.4f}   (VQC baseline: 0.3028)")
print(f"  Decision threshold: {best_t:.4f}")
print(f"  Confusion Matrix: TN={cm[0,0]} FP={cm[0,1]} "
      f"FN={cm[1,0]} TP={cm[1,1]}")
print()
print(classification_report(y_test, test_preds,
      target_names=["No Fire", "Wildfire"], digits=3))

print("\n  ── Circuit cost summary ──────────────────────────────────────")
print(f"  Stage 1 (classical SVM)     : 0 quantum circuits")
print(f"  Stage 2 (kernel matrix)     : {n_sv*n_sv//2:,} circuits  "
      f"({n_sv}×{n_sv}/2)")
print(f"  Stage 4 val kernel          : {len(X_val_q)*n_sv:,} circuits")
print(f"  Stage 4 test kernel         : {len(X_test_q)*n_sv:,} circuits")
total = n_sv*n_sv//2 + (len(X_val_q)+len(X_test_q))*n_sv
print(f"  Total circuits              : {total:,}")
print(f"  SV1 cost estimate           : "
      f"${total*0.00375:.2f} (at $0.00375/task min)")
print(f"  IBM Open Plan               : fits in 10-min free quota")

# Save results
results = {
    "model":            "Hybrid QSVM v9 (Classical Filter + ZZFeatureMap Kernel)",
    "feature_map":      f"ZZFeatureMap (N_QUBITS={N_QUBITS}, N_REPS={N_REPS})",
    "n_train":          len(X_train_q),
    "n_sv_stage1":      int(n_sv),
    "n_sv_final":       int(svm_qk.n_support_.sum()),
    "kernel_circuits":  int(n_sv*n_sv//2),
    "decision_threshold": round(float(best_t), 4),
    "test_accuracy":    round(float(acc), 4),
    "roc_auc":          round(float(auc), 4),
    "f1_score":         round(float(f1), 4),
    "confusion_matrix": cm.tolist(),
    "vqc_roc_auc_baseline": 0.6710,
    "vqc_f1_baseline":      0.3028,
}
os.makedirs("results_qsvm", exist_ok=True)
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n  Results saved → {RESULTS_PATH}")



  HYBRID QSVM — FINAL RESULTS
  Accuracy        : 61.49%
  ROC-AUC         : 0.6271   (VQC baseline: 0.6710)
  F1 Score        : 0.2369   (VQC baseline: 0.3028)
  Decision threshold: 0.2500
  Confusion Matrix: TN=1152 FP=749 FN=50 TP=124

              precision    recall  f1-score   support

     No Fire      0.958     0.606     0.743      1901
    Wildfire      0.142     0.713     0.237       174

    accuracy                          0.615      2075
   macro avg      0.550     0.659     0.490      2075
weighted avg      0.890     0.615     0.700      2075


  ── Circuit cost summary ──────────────────────────────────────
  Stage 1 (classical SVM)     : 0 quantum circuits
  Stage 2 (kernel matrix)     : 5,547,780 circuits  (3331×3331/2)
  Stage 4 val kernel          : 6,911,825 circuits
  Stage 4 test kernel         : 6,911,825 circuits
  Total circuits              : 19,371,430
  SV1 cost estimate           : $72642.86 (at $0.00375/task min)
  IBM Open Plan               : fits in 

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 2023 WILDFIRE RISK PREDICTIONS
# ─────────────────────────────────────────────────────────────────────────────


print("\nRunning 2023 risk predictions…")
K_pred = build_kernel_predict(X_pred_q, X_sv, label="2023")
pred_probs = svm_qk.predict_proba(K_pred)[:, 1]

df_out = df_2023.copy()
df_out["risk_probability"]   = pred_probs
df_out["risk_level"]         = pd.cut(
    pred_probs, bins=[0, 0.33, 0.55, 0.70, 1.0],
    labels=["Low", "Moderate", "High", "Extreme"])
df_out["wildfire_predicted"] = (pred_probs >= best_t).astype(int)

at_risk = df_out["wildfire_predicted"].sum()
print(f"  ZIPs at fire risk 2023: {at_risk} / {len(df_out)}")
print(f"  Prob range: [{pred_probs.min():.3f}, {pred_probs.max():.3f}]")

print(f"\n  {'ZIP':>8}  {'Prob':>8}  {'Level':>10}  {'Risk?':>6}")
print("  " + "-"*40)
for _, r in df_out.sort_values("risk_probability", ascending=False).head(20).iterrows():
    print(f"  {int(r['zip']):>8}  {r['risk_probability']:>8.3f}  "
          f"{str(r['risk_level']):>10}  "
          f"{'Yes' if r['wildfire_predicted'] else 'No':>6}")




Running 2023 risk predictions…
   2023 [ 100/2593] | 333100 circuits | 2882s
   2023 [ 200/2593] | 666200 circuits | 4619s
   2023 [ 300/2593] | 999300 circuits | 6380s
   2023 [ 400/2593] | 1332400 circuits | 8127s
   2023 [ 500/2593] | 1665500 circuits | 9665s
   2023 [ 600/2593] | 1998600 circuits | 11058s
   2023 [ 700/2593] | 2331700 circuits | 12443s
   2023 [ 800/2593] | 2664800 circuits | 13952s
   2023 [ 900/2593] | 2997900 circuits | 15360s
   2023 [1000/2593] | 3331000 circuits | 16807s
   2023 [1100/2593] | 3664100 circuits | 18409s
   2023 [1200/2593] | 3997200 circuits | 19902s
   2023 [1300/2593] | 4330300 circuits | 21346s
   2023 [1400/2593] | 4663400 circuits | 22791s
   2023 [1500/2593] | 4996500 circuits | 24212s
   2023 [1600/2593] | 5329600 circuits | 25642s
   2023 [1700/2593] | 5662700 circuits | 27033s
   2023 [1800/2593] | 5995800 circuits | 28438s
   2023 [1900/2593] | 6328900 circuits | 30084s
   2023 [2000/2593] | 6662000 circuits | 31896s
   2023 [2100/25

In [17]:
import json

df_out["risk_level"] = df_out["risk_level"].astype(str)  # Convert Categorical → str

records = df_out[["zip", "risk_probability", "risk_level", "wildfire_predicted"]].to_dict(orient="records")

# ── Load existing model results and merge predictions under correct key ────────
results_path = RESULTS_PATH  # "results_qsvm/qsvm_results_v9.json"
try:
    with open(results_path) as f:
        full_results = json.load(f)
except FileNotFoundError:
    full_results = {}

full_results["predictions_2023"] = records  # key matches what plotting scripts expect

with open(results_path, "w") as f:
    json.dump(full_results, f, indent=2)

print(f"Saved {len(records)} ZIP predictions → {results_path} (key: predictions_2023)")

# Also save standalone predictions file
with open("wildfire_predictions_2023.json", "w") as f:
    json.dump(records, f, indent=2)
print("Also saved → wildfire_predictions_2023.json")

Saved 2593 ZIP predictions → results_qsvm/qsvm_results_v9.json (key: predictions_2023)
Also saved → wildfire_predictions_2023.json
